In [ ]:
import bs4
import matplotlib
from selenium import webdriver
import selenium
from selenium.webdriver.chrome.options import Options
import time
import re
import pandas as pd
import datetime as dt

Uporabili bomo knjižico Selenium, saj requests modul ni deloval zaradi IMDb-jeve zaščite

In [2]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_argument("--headless") # ne pokaže brksalnika


Uporabili bomo modul Beautiful Soup, da lahko lepše analiziramo podatke. Ti podatki bodo vedno posodobljeni, saj se bodo s posodobitvijo spletne strani posodobili tudi naši podatki.

In [3]:
def preveri_st_filmov(url):
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(15) # da se stran naloži
    piskoti = driver.find_element(selenium.webdriver.common.by.By.XPATH, "//html/body/div[2]/div[2]/div/div[2]/div/button[2]")
    piskoti.click()
    
    vsebina = bs4.BeautifulSoup(driver.page_source, 'html.parser')
    filmi = vsebina.find("ul", class_="ipc-metadata-list")
    
    seznam_filmov = filmi.find_all(class_="ipc-metadata-list-summary-item")
    driver.quit()
    return seznam_filmov, len(seznam_filmov)

In [ ]:
def iskanje_filmov(zacetno_leto_iskanja, koncno_leto_iskanja, velikost_vzorca, izbrani_zanri):   
    # link do naprednega iskanja filmov (analizirali bomo le filme). Parameter count predstavlja velikost podatkov
    # podatki za zanr bodo oblike https://www.imdb.com/search/title/?title_type=feature&genres=zanr1,zanr2,...
    seznam_moznih_zanrov = ['action', 'adventure', 'animation', 'biography', 'comedy', 'crime', 'documentary', 'drama', 'family', 'fantasy', 'film-noir', 'game-show', 'history', 'horror', 'music', 'musical', 'mystery', 'news', 'reality-tv', 'romance', 'sci-fi', 'sport', 'talk-show', 'thriller', 'war', 'western']
        
    veljaven_izbor = False 
    najdeni_filmi = []

    # preveri ali je izbor zanra veljaven
    while not veljaven_izbor:
        izbrani_zanri.rstrip() # da se izognemo nepotrebnim presledkom na koncu vrstice
        izbrani_zanri = izbrani_zanri.split(",")
        izbrani_zanri = [x.lower() for x in izbrani_zanri]

        if set(izbrani_zanri).issubset(seznam_moznih_zanrov):
            veljaven_izbor = True
            url_izbor = ",".join(izbrani_zanri)
            nov_url = f"https://www.imdb.com/search/title/?title_type=feature&release_date={zacetno_leto_iskanja}-01-01,{koncno_leto_iskanja}-12-31&genres={url_izbor}&count={velikost_vzorca}"
            # rabimo preveriti ali je izbor preozek ali ne
            filmi, st_filmov = preveri_st_filmov(nov_url)
            if st_filmov > 0:
                najdeni_filmi.extend(filmi)
            elif st_filmov == 0:
                print("Izbran nabor je preozek. Poskusi izbrati širši izbor")
                veljaven_izbor = False
            
        else:
            print("To ni veljaven izbor vrste filmov.")
        
        
        return najdeni_filmi, zacetno_leto_iskanja, koncno_leto_iskanja
    # zdaj imamo veljaven nabor žanrov, po kateri lahko naredimo analizo podatkov
    
    


Zdaj, ko imamo podakte o filmih, moramo izluščiti podatke o vsakem filmu (naslov, dolžina, leto izdaje, ocena, ...)

In [18]:
#funkcija, ki zapise podatke v datoteko

def zapisi_podatke(zac_leto, kon_leto, velikost_vzorca, vrsta_filmov):
    nas_seznam_filmov, zac_leto, kon_leto = iskanje_filmov(zac_leto, kon_leto, velikost_vzorca, vrsta_filmov)
    podatki_o_filmih = []
    ime_datoteke =  f"Filmi_{vrsta_filmov}_{zac_leto}-{kon_leto}.txt"
    
    for film in nas_seznam_filmov:
        neobdelan_naslov = re.findall(r'<h4 class="ipc-title__text">(.+?)</h4>', film.decode_contents())[0]
        naslov = re.sub(r"\d+. ", "", neobdelan_naslov)
        dodatni_podatki = re.findall(r'<li class="ipc-inline-list__item" role="presentation">(.+?)</li>', film.decode_contents())
        
        # problem je, da nekateri filmi še nimajo določene dolžine
        leto_izida = dodatni_podatki[0]
        try:
            neobdelano_trajanje = re.findall(r"\d+", dodatni_podatki[1])
            try:
                trajanje = int(neobdelano_trajanje[0]) * 60 + int(neobdelano_trajanje[1]) # h * 60 + min
            except IndexError:
                trajanje = int(neobdelano_trajanje[0]) # če imamo samo 1 element imamo samo minute
        except IndexError:
            trajanje = "N/A"
        
        # filmi, ki še niso bili izdani še nimajo ocene
        try:
            ocena = re.findall(r'<span class="ipc-rating-star--rating">(.+?)</span>', film.decode_contents())[0]
        except IndexError:
            ocena = "NE"

        podatki_o_filmih.append([naslov, leto_izida, trajanje, ocena])
    with open(ime_datoteke, "w", encoding="utf-8") as file:
        file.write(f"naslov, leto_izida, trajanje, ocena\n")
        for podatek in podatki_o_filmih:
            file.write(f"{podatek[0]}, {podatek[1]}, {podatek[2]}, {podatek[3]}\n")
            

zapisi_podatke(2000, 2024, 250, "action,adventure,comedy")
        
        